### This tutorial deals with reading an SU data set and creating a SEG-Y file

#### Download this data through the link: https://wiki.seg.org/wiki/2D_Vibroseis_Line_001#Download_Link

In [1]:
import logging 
import numpy as np 
import sys
import seisio

In [2]:
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s', force=True)
log=logging.getLogger("main")

In [3]:
infile = "/media/ashraf/𝓐𝓢𝓗𝓡𝓐𝓕1/100-DAYS-CHALLENGE-DATA/2D_Land_vibro_data_2ms/Line_001.su"          # SU file, IEEE floats, little endian
outfile = "/media/ashraf/𝓐𝓢𝓗𝓡𝓐𝓕1/100-DAYS-CHALLENGE-DATA/2D_Land_vibro_data_2ms/convertedLine_001.sgy"   # SEGY file, IEEE floats, big endian

In [4]:
# Create seisio object to read the input SU file.
sin=seisio.input(infile)

INFO: Input file: /media/ashraf/𝓐𝓢𝓗𝓡𝓐𝓕1/100-DAYS-CHALLENGE-DATA/2D_Land_vibro_data_2ms/Line_001.su
INFO: Assuming fixed-length traces for SU data.
INFO: Data sample format: 4-byte IEEE floating-point.
INFO: Input file endianess looks to be '<' (best guess).
INFO: Byte offset of first trace relative to start of file: 0 bytes.
INFO: Number of samples per data trace: 1501.
INFO: Sampling interval: 2000 (unit as per SU standard).
INFO: Delay (on first trace): 0 (unit as per SU standard).
INFO: Number of data traces in file: 71284.


In [5]:
sout=seisio.output(outfile, ns=sin.ns, vsi=sin.vsi, endian=">", format=5, txtenc="ebcdic")

INFO: Output file: /media/ashraf/𝓐𝓢𝓗𝓡𝓐𝓕1/100-DAYS-CHALLENGE-DATA/2D_Land_vibro_data_2ms/convertedLine_001.sgy
INFO: Output number of samples per data trace: 1501.
INFO: Output data sample format: 4-byte IEEE floating-point.
INFO: Output file endianess set to '>'.
INFO: Number of additional textual header records: 0.
INFO: Byte offset of first trace relative to start of file: 3600 bytes.
INFO: Number of trailer stanza records: 0.
INFO: SEG-Y trace header extension 1 is not present.
INFO: Number of user-defined trace headers: 0
INFO: Creating file according to SEG-Y rev. 1.0.


##### SEG-Y files require (at least) a primary textual file header (3200 bytes) and a binary file header (400 bytes). We can get templates and fill them in ourselves, or we let seisio create default (minimal) file headers for us similar to SU's segyhdrs program.

In [11]:
textual_template = sout.txthead_template
binary_template = sout.binhead_template

In [12]:
sout.init()

INFO: SEG-Y textual file header encoding set to 'EBCDIC'.
INFO: Wrote textual and binary file headers and 0 add. header record(s).


In [13]:
log.info("Size of output file now %d bytes (should be 3600).", sout.fsize)

INFO: Size of output file now 3600 bytes (should be 3600).


Calling the init() method several times is ignored:

In [14]:
sout.init()

####  Now we read the whole data set from the SU file but we could obviously also loop through the data in case of very large files.

In [15]:
dataset=sin.read_dataset()

INFO: Reading entire file (71284 traces) from disk...
INFO: Reading all traces took 0.3 seconds.


In [17]:
#  Before writing the data to disk, let's sort the data in some way 
# (just for demonstrating the feature) in descending order
dataset.sort(order="cdp", kind="stable")

In [18]:
dataset=np.sort(dataset, order="cdp", kind="stable")

In [20]:
dataset = np.sort(dataset, order=["tracl"])[::-1]

In [21]:
# Write the data to disk. This will also update the textual and binary headers based on the data.
nwritten=sout.write_traces(traces=dataset)

INFO: Swapping bytes of output data.
INFO: Writing 71284 trace(s) to disk...


In [22]:
sout.finalize()

INFO: Finalizing output file and re-writing updated binary header.
INFO: Wrote a total of 71284 trace(s), file size: 445100896 bytes.


In [23]:
try:
    sout.write_traces(traces=dataset)
except RuntimeError:
    log.info("Caught 'RuntimeError' exception as expected.")

INFO: Caught 'RuntimeError' exception as expected.
